# Age and Gender Prediction using Transfer Learning

This notebook builds a complete end-to-end deep learning project for predicting **age** and **gender** from face images using the **UTKFace dataset** and **MobileNetV2 transfer learning**.

The model has two outputs:

- **Age prediction**: regression problem
- **Gender prediction**: binary classification problem

## Cell 1: Install and import required libraries

This cell imports all required libraries for downloading data, loading images, building the model, training, evaluation, visualization, and prediction.

In [ ]:
import os
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import userdata
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("TensorFlow version:", tf.__version__)

## Cell 2: Set random seed

Setting a seed helps make the experiment more reproducible. It controls randomness in NumPy, Python, and TensorFlow.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Cell 3: Kaggle secret-key method

Before running this cell, add these two secrets in Google Colab:

- `KAGGLE_USERNAME`
- `KAGGLE_KEY`

Go to **Colab left panel → Secrets/key icon → Add new secret**. This method is safer than uploading `kaggle.json`.

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

if os.environ["KAGGLE_USERNAME"] is None or os.environ["KAGGLE_KEY"] is None:
    raise ValueError("Please add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets.")

print("Kaggle secrets loaded successfully.")

## Cell 4: Download UTKFace dataset

This cell downloads the UTKFace dataset from Kaggle. UTKFace image filenames contain labels in this format:

`age_gender_race_date.jpg`

Example: `25_0_0_201701.jpg` means age 25 and gender 0.

In [ ]:
!kaggle datasets download -d jangedoo/utkface-new -p /content --force

## Cell 5: Extract dataset

The downloaded file is a ZIP file. This cell extracts it into the `/content` directory.

In [ ]:
zip_path = "/content/utkface-new.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")

## Cell 6: Define dataset path

After extraction, UTKFace images are usually stored in this folder.

In [ ]:
DATASET_DIR = "/content/utkface_aligned_cropped/UTKFace"

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError("Dataset folder not found. Please check extraction path.")

print("Total files:", len(os.listdir(DATASET_DIR)))

## Cell 7: Create dataframe from image filenames

This cell reads filenames and extracts age and gender labels.

Gender labels in UTKFace:

- `0 = Male`
- `1 = Female`

In [ ]:
data = []

for filename in os.listdir(DATASET_DIR):
    try:
        parts = filename.split("_")
        age = int(parts[0])
        gender = int(parts[1])
        filepath = os.path.join(DATASET_DIR, filename)
        data.append([filepath, age, gender])
    except Exception:
        continue

df = pd.DataFrame(data, columns=["filepath", "age", "gender"])
print(df.shape)
df.head()

## Cell 8: Basic dataset analysis

This cell checks the distribution of age and gender labels. This helps us understand whether the dataset is balanced or imbalanced.

In [ ]:
print("Age range:", df["age"].min(), "to", df["age"].max())
print("Gender distribution:")
print(df["gender"].value_counts())

df["gender_name"] = df["gender"].map({0: "Male", 1: "Female"})
df["gender_name"].value_counts().plot(kind="bar", title="Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

df["age"].plot(kind="hist", bins=30, title="Age Distribution")
plt.xlabel("Age")
plt.show()

## Cell 9: Show sample images

This cell displays a few face images with their age and gender labels.

In [ ]:
sample_df = df.sample(9, random_state=SEED)
plt.figure(figsize=(10, 10))

for i, row in enumerate(sample_df.itertuples(), 1):
    img = load_img(row.filepath, target_size=(160, 160))
    plt.subplot(3, 3, i)
    plt.imshow(img)
    plt.title(f"Age: {row.age}, Gender: {row.gender_name}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Cell 10: Train-validation-test split

The dataset is divided into three parts:

- Training data: used to train the model
- Validation data: used to monitor performance during training
- Test data: used for final evaluation

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["gender"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["gender"])

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

## Cell 11: Image loading function

This function reads an image file, resizes it, converts it into a tensor, and applies MobileNetV2 preprocessing. It returns the image with two labels: age and gender.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image_and_labels(filepath, age, gender):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)

    labels = {
        "age_output": tf.cast(age, tf.float32),
        "gender_output": tf.cast(gender, tf.float32)
    }

    return image, labels

## Cell 12: Create TensorFlow datasets

This modern `tf.data` method avoids the old `class_mode="multi_output"` generator error. It is stable with recent TensorFlow versions.

In [ ]:
def make_dataset(dataframe, shuffle=False):
    filepaths = dataframe["filepath"].values
    ages = dataframe["age"].values
    genders = dataframe["gender"].values

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, ages, genders))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=SEED)

    dataset = dataset.map(load_image_and_labels, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

print("Datasets created successfully.")

## Cell 13: Build transfer learning model

We use MobileNetV2 pretrained on ImageNet as the feature extractor.

The model has two heads:

- `age_output`: predicts age using linear activation
- `gender_output`: predicts gender using sigmoid activation

In [ ]:
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomZoom(0.10)
], name="data_augmentation")

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_tensor=x
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.30)(x)

age_output = Dense(1, activation="linear", name="age_output")(x)
gender_output = Dense(1, activation="sigmoid", name="gender_output")(x)

model = Model(inputs=inputs, outputs=[age_output, gender_output])
model.summary()

## Cell 14: Compile model

Age prediction is a regression task, so we use MAE loss. Gender prediction is a binary classification task, so we use binary crossentropy.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        "age_output": "mae",
        "gender_output": "binary_crossentropy"
    },
    metrics={
        "age_output": ["mae"],
        "gender_output": ["accuracy"]
    },
    loss_weights={
        "age_output": 1.0,
        "gender_output": 5.0
    }
)

## Cell 15: Define callbacks

Callbacks help stop training early and save the best model automatically.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_age_gender_model.keras",
    monitor="val_loss",
    save_best_only=True
)

## Cell 16: Train the model

This cell trains the frozen MobileNetV2 feature extractor with custom output heads.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

## Cell 17: Plot training results

This cell plots age error and gender accuracy over epochs.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["age_output_mae"], label="Train Age MAE")
plt.plot(history.history["val_age_output_mae"], label="Validation Age MAE")
plt.title("Age Prediction MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["gender_output_accuracy"], label="Train Gender Accuracy")
plt.plot(history.history["val_gender_output_accuracy"], label="Validation Gender Accuracy")
plt.title("Gender Prediction Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Cell 18: Evaluate model on test data

This cell checks final model performance on unseen test images.

In [ ]:
test_results = model.evaluate(test_ds)
print(test_results)

## Cell 19: Fine-tuning

After initial training, we unfreeze the last few layers of MobileNetV2 and train with a very small learning rate. This can improve performance.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        "age_output": "mae",
        "gender_output": "binary_crossentropy"
    },
    metrics={
        "age_output": ["mae"],
        "gender_output": ["accuracy"]
    },
    loss_weights={
        "age_output": 1.0,
        "gender_output": 5.0
    }
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, checkpoint]
)

## Cell 20: Save final model

This cell saves the trained model for later use.

In [ ]:
model.save("final_age_gender_prediction_model.keras")
print("Model saved successfully.")

## Cell 21: Single image prediction function

This function loads one image and predicts age and gender.

In [ ]:
def predict_age_gender(image_path):
    image = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    image_array = img_to_array(image)
    image_array = preprocess_input(image_array)
    image_array = np.expand_dims(image_array, axis=0)

    age_pred, gender_pred = model.predict(image_array)

    predicted_age = round(float(age_pred[0][0]))
    predicted_gender = "Female" if float(gender_pred[0][0]) >= 0.5 else "Male"

    plt.imshow(load_img(image_path))
    plt.axis("off")
    plt.title(f"Predicted Age: {predicted_age}, Gender: {predicted_gender}")
    plt.show()

    return predicted_age, predicted_gender

## Cell 22: Test prediction on random image

This cell selects one random test image and predicts age and gender.

In [ ]:
sample = test_df.sample(1, random_state=SEED).iloc[0]
print("Actual Age:", sample["age"])
print("Actual Gender:", "Female" if sample["gender"] == 1 else "Male")

predict_age_gender(sample["filepath"])

## Cell 23: Upload your own image in Colab

Run this cell to upload your own face image and test the model.

In [ ]:
from google.colab import files

uploaded = files.upload()

for uploaded_file in uploaded.keys():
    predict_age_gender(uploaded_file)

## Final Notes

This project demonstrates a complete multi-output transfer learning system. It avoids the common Keras error:

`output_signature must contain objects that are subclass of tf.TypeSpec but found list`

This error is avoided because we used `tf.data.Dataset` and dictionary-based labels instead of the old `ImageDataGenerator` multi-output method.